# Chấm điểm động tác chống đẩy — Bước 2: đếm rep và cắt rep

Đầu vào: thư mục `pushup_keypoints` do notebook bước 1 tạo ra (`keypoints/*.npz` + `pushup_index.csv`).

Đầu ra: `pushup_reps.csv`, mỗi dòng là một rep (video nào, khung bắt đầu, khung thấp nhất, khung kết thúc, có dùng được không).

Phần này là **luật hình học + xử lý tín hiệu, không phải Deep Learning**.

## 1. Tìm thư mục keypoint

Tự tìm trong `/kaggle/working` (cùng phiên với bước 1) hoặc `/kaggle/input` (đã tạo Kaggle Dataset).

In [ ]:
import os, glob

found = glob.glob("/kaggle/working/**/pushup_index.csv", recursive=True) \
      + glob.glob("/kaggle/input/**/pushup_index.csv", recursive=True)
assert found, "Không thấy pushup_index.csv. Hãy chạy notebook bước 1 hoặc Add Input dataset keypoint."

KP_ROOT = os.path.dirname(found[0])
OUT_DIR = "/kaggle/working/pushup_reps"
os.makedirs(OUT_DIR, exist_ok=True)
print("Keypoint:", KP_ROOT)

## 2. Hàm tạo tín hiệu và đếm rep

**Tín hiệu lên/xuống** (giá trị cao = đang ở trên, thấp = đang ở dưới):
- Góc ngang, góc chéo: góc khuỷu (vai–khuỷu–cổ tay) của bên tay nhìn rõ hơn.
- Góc trước: độ cao vai so với cổ tay, chia cho độ rộng vai (góc khuỷu bị méo phối cảnh).

**Khung hợp lệ**: phát hiện được người, các khớp dùng để tính có visibility > 0.5 và nằm trong ảnh. Khung không hợp lệ được nội suy.

**Đếm rep**: mỗi đáy của tín hiệu là một rep. Không dùng ngưỡng góc cố định, vì mỗi người xuống sâu khác nhau;
đáy phải nổi bật ít nhất 35% biên độ của chính video đó. Rep kéo dài từ đỉnh trước đáy tới đỉnh sau đáy.

Rep bị loại (`ok = False`) nếu quá ngắn/quá dài, dưới 70% khung hợp lệ, hoặc **biên độ quá nhỏ** (`MIN_AMP`).
Video có dưới 30% khung hợp lệ thì bỏ cả video.

Vì sao cần biên độ tối thiểu: khi người tập đứng yên ở tư thế chờ, tín hiệu chỉ còn nhiễu và "đáy nổi bật so với chính video đó"
vẫn tìm ra rep giả. Ngưỡng góc trước (0.10 lần độ rộng vai) được chọn từ dữ liệu QEVD: rep giả ở tư thế chờ có biên độ khoảng 0.03,
rep thật nông nhất khoảng 0.15. Ngưỡng góc ngang/chéo (3°) mới là giá trị tạm, vì chưa có video người đứng yên quay ngang để hiệu chỉnh.

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, savgol_filter

VIS_THR = 0.5
MIN_AMP = {"front": 0.10, "side": 3.0, "diagonal": 3.0}     # biên độ tối thiểu của một rep, theo đơn vị của tín hiệu
L_SH, R_SH, L_EL, R_EL, L_WR, R_WR = 11, 12, 13, 14, 15, 16
L_HIP, L_ANK = 23, 27


def joint_angle(a, b, c):
    ba, bc = a - b, c - b
    cos = (ba * bc).sum(-1) / (np.linalg.norm(ba, axis=-1) * np.linalg.norm(bc, axis=-1) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))


def fill_gaps(x, valid):
    # Nội suy tuyến tính các khung không hợp lệ
    x = x.astype(np.float64).copy()
    idx = np.arange(len(x))
    if valid.sum() >= 2:
        x[~valid] = np.interp(idx[~valid], idx[valid], x[valid])
    return x


def near_side(vis):
    # 0 = bên trái, 1 = bên phải: bên nào tay nhìn rõ hơn
    return 0 if vis[:, [L_SH, L_EL, L_WR]].mean() >= vis[:, [R_SH, R_EL, R_WR]].mean() else 1


def rep_signal(d, view):
    kp = d["kp2d"]
    px = kp[:, :, :2] * np.array([int(d["width"]), int(d["height"])])   # về pixel để góc không bị méo
    vis, det = kp[:, :, 3], d["detected"]

    def in_frame(j):
        return (kp[:, j, 0] > -0.1) & (kp[:, j, 0] < 1.1) & (kp[:, j, 1] > -0.1) & (kp[:, j, 1] < 1.1)

    if view == "front":
        sh_y = px[:, [L_SH, R_SH], 1].mean(1)
        wr_y = px[:, [L_WR, R_WR], 1].mean(1)
        sh_w = np.linalg.norm(px[:, L_SH] - px[:, R_SH], axis=-1)
        valid = det & (vis[:, [L_SH, R_SH]].min(1) > VIS_THR) & (vis[:, [L_WR, R_WR]].max(1) > VIS_THR) \
            & in_frame(L_SH) & in_frame(R_SH)
        scale = np.median(sh_w[valid]) if valid.any() else 1.0
        sig = (wr_y - sh_y) / (scale + 1e-6)
    else:
        side = near_side(vis)
        s, e, w = L_SH + side, L_EL + side, L_WR + side
        sig = joint_angle(px[:, s], px[:, e], px[:, w])
        valid = det & (vis[:, [s, e, w]].min(1) > VIS_THR) & in_frame(s) & in_frame(e) & in_frame(w)

    sig = fill_gaps(sig, valid)
    if len(sig) >= 7:
        sig = savgol_filter(sig, 7, 2)      # làm mượt nhẹ
    return sig, valid


def find_reps(sig, valid, fps, min_amp=0.0, min_prom=0.35, min_rep_s=0.8, max_rep_s=8.0, min_valid=0.7):
    if valid.mean() < 0.3:      # video gần như không nhìn rõ tay: bỏ cả video
        return []
    lo, hi = np.percentile(sig[valid], [5, 95])
    rng = hi - lo
    bottoms, _ = find_peaks(-sig, prominence=min_prom * rng, distance=max(1, int(min_rep_s * fps * 0.6)))
    half = int(max_rep_s * fps / 2)
    reps = []
    for i, b in enumerate(bottoms):
        left = bottoms[i - 1] if i > 0 else max(0, b - half)
        right = bottoms[i + 1] if i + 1 < len(bottoms) else min(len(sig) - 1, b + half)
        start = left + int(np.argmax(sig[left:b + 1]))
        end = b + int(np.argmax(sig[b:right + 1]))
        dur = (end - start) / fps
        amp = float(min(sig[start], sig[end]) - sig[b])       # tín hiệu hạ xuống bao nhiêu so với đỉnh thấp hơn
        deep = amp > 0.25 * rng and amp >= min_amp
        valid_frac = float(valid[start:end + 1].mean()) if end > start else 0.0
        reps.append(dict(start=int(start), bottom=int(b), end=int(end), dur=round(dur, 2),
                         valid_frac=round(valid_frac, 2), amp=round(amp, 3),
                         ok=bool(min_rep_s <= dur <= max_rep_s and deep and valid_frac >= min_valid)))
    return reps

## 3. Chạy trên toàn bộ video

Ngoài vị trí rep, lưu thêm hai số đo tại khung thấp nhất để phân tích ở mục 6:
- `elbow_bottom`: góc khuỷu (nhỏ = xuống sâu)
- `hip_bottom`: góc vai–hông–cổ chân (180° = thân thẳng). Để trống nếu hông/cổ chân không nhìn thấy (góc trước).

In [ ]:
index = pd.read_csv(os.path.join(KP_ROOT, "pushup_index.csv"))

rows = []
for _, r in index.iterrows():
    d = np.load(os.path.join(KP_ROOT, "keypoints", r.file))
    sig, valid = rep_signal(d, r["view"])
    kp = d["kp2d"]
    px = kp[:, :, :2] * np.array([int(d["width"]), int(d["height"])])
    vis = kp[:, :, 3]
    side = near_side(vis)
    elbow = joint_angle(px[:, L_SH + side], px[:, L_EL + side], px[:, L_WR + side])
    hip = joint_angle(px[:, L_SH + side], px[:, L_HIP + side], px[:, L_ANK + side])
    hip[vis[:, [L_SH + side, L_HIP + side, L_ANK + side]].min(1) < VIS_THR] = np.nan

    for k, rep in enumerate(find_reps(sig, valid, float(d["fps"]), MIN_AMP[r["view"]])):
        rows.append(dict(file=r.file, subject=r.subject, quality=r.quality, view=r["view"], rep_id=k, **rep,
                         elbow_bottom=round(float(elbow[rep["bottom"]]), 1),
                         hip_bottom=round(float(hip[rep["bottom"]]), 1)))

reps = pd.DataFrame(rows)
reps.to_csv(os.path.join(OUT_DIR, "pushup_reps.csv"), index=False)
print("Tổng số rep tìm được:", len(reps), "| dùng được:", int(reps.ok.sum()))

## 4. Thống kê số rep

In [ ]:
print("Số rep dùng được theo góc quay × chất lượng:")
display(reps[reps.ok].groupby(["view", "quality"]).size().unstack("quality"))

print("Số rep dùng được của từng video (đối chiếu với số đếm tay):")
counts = reps[reps.ok].pivot_table(index=["subject", "quality"], columns="view", values="rep_id",
                                   aggfunc="count", fill_value=0)
counts.to_csv(os.path.join(OUT_DIR, "rep_counts_per_video.csv"))
display(counts)

## 5. Kiểm tra bằng mắt

Đường xanh là tín hiệu lên/xuống. Tam giác xanh lá: rep dùng được. Tam giác đỏ: rep bị loại. Chấm đen: khung không hợp lệ (đã nội suy).

Đổi `SUBJECT` để xem người khác. Nên xem hết 24 người một lượt.

In [ ]:
import matplotlib.pyplot as plt

SUBJECT = 4

fig, axes = plt.subplots(2, 3, figsize=(22, 7))
for r_, quality in enumerate(["good", "bad"]):
    for c_, view in enumerate(["front", "side", "diagonal"]):
        name = f"subject_{SUBJECT:03d}_push_up_{quality}_{view}.npz"
        d = np.load(os.path.join(KP_ROOT, "keypoints", name))
        sig, valid = rep_signal(d, view)
        found_reps = find_reps(sig, valid, float(d["fps"]), MIN_AMP[view])
        ax = axes[r_, c_]
        ax.plot(sig)
        ax.plot(np.where(~valid)[0], sig[~valid], "k.", ms=3)
        for rep in found_reps:
            ax.plot(rep["bottom"], sig[rep["bottom"]], "v", ms=9, color="green" if rep["ok"] else "red")
            ax.axvline(rep["start"], color="gray", alpha=0.3)
        ax.set_title(f"{quality}/{view}: {sum(x['ok'] for x in found_reps)} rep")
        ax.set_xlabel("khung hình")
        ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. "Bad" là lỗi gì? So sánh good/bad ở góc ngang

Dataset không nói rõ "bad" là lỗi gì. Bảng dưới so sánh trung vị của từng người ở góc ngang:
- `elbow_bottom` của bad lớn hơn good nhiều → lỗi **xuống không đủ sâu**
- `hip_bottom` của bad nhỏ hơn good nhiều → lỗi **hông võng / mông nhô**
- Hai số gần như nhau → lỗi không thấy được ở góc ngang, cần mở video xem

In [ ]:
side = reps[reps.ok & (reps["view"] == "side")]
cmp = side.groupby(["subject", "quality"])[["elbow_bottom", "hip_bottom"]].median().unstack("quality").round(0)
cmp["elbow_diff"] = cmp[("elbow_bottom", "bad")] - cmp[("elbow_bottom", "good")]
cmp["hip_diff"] = cmp[("hip_bottom", "bad")] - cmp[("hip_bottom", "good")]
display(cmp)

## 7. Kết quả

Tải `pushup_reps.csv` về (thư mục `/kaggle/working/pushup_reps`), hoặc Save Version để giữ lại.

In [ ]:
!ls -lh "{OUT_DIR}"